# 15.1 基于内容的推荐 / Content-Based Recommendation

**中文**：本节我们从最直观的推荐范式开始——**基于内容（content-based）**。核心思想一句话：*"你喜欢过的东西，再给你推相似的东西"*。相似与否由**物品自身的特征**（电影的类型、文本描述、标签）决定，完全不需要别的用户的数据。
**English**: We start with the most intuitive recommendation paradigm — **content-based** filtering. One sentence: *"recommend items similar to what you already liked."* Similarity is judged purely from the **item's own features** (a movie's genres, text, tags) — no other users' data is needed.

---

**中文**：为什么先讲它？因为它是推荐系统的"Hello World"，而且直接复用了我们在 Part 11（经典 NLP）学过的 **TF-IDF + 余弦相似度**。学完你会对一个面试高频问题有标准答案：*"协同过滤和基于内容有什么区别？冷启动怎么办？"*
**English**: Why first? It is the "Hello World" of recommenders and directly reuses the **TF-IDF + cosine similarity** from Part 11 (Classic NLP). By the end you will have a crisp answer to a very common interview question: *"What is the difference between collaborative filtering and content-based, and how do they handle cold start?"*

> 💡 **面试速查 / Interview cheat-sheet（★★★ 出镜率极高）**
> **中文**：基于内容 = 用**物品特征**算相似度 → 推相似物品；优点：**无冷启动物品问题**（新电影只要有类型标签就能推）、可解释（"因为你看过同类型"）；缺点：**新用户冷启动**（不知道你喜欢啥）、**过度专门化**（filter bubble，永远只推同一类）、特征工程依赖人工。
> **English**: Content-based = compute similarity from **item features** → recommend similar items. Pros: **no item cold-start** (a new movie with genre tags is immediately recommendable), explainable ("because you watched the same genre"). Cons: **user cold-start** (we don't yet know your taste), **over-specialization** (filter bubble), and reliance on hand-crafted features.


In [ ]:

# ============================================================
# 环境与数据：MovieLens-100k / Setup & data
# 中文：MovieLens 是推荐系统领域最经典的公开数据集（GroupLens 实验室出品）。
#       100k 版本含 943 用户对 1682 部电影的 10 万条 1~5 星评分，且每部电影带 19 个类型标签。
# English: MovieLens is THE classic public recsys dataset (from the GroupLens lab).
#          The 100k version: 943 users, 1682 movies, 100k ratings (1-5 stars),
#          and each movie carries 19 binary genre tags — perfect for content-based.
# ============================================================
import os, numpy as np, pandas as pd
np.random.seed(0)                                  # 固定随机种子，结果可复现 / reproducible

R = os.path.expanduser("~/.cache/dsfs_recsys/ml-100k")   # 数据缓存目录 / cache dir

# --- 评分表 u.data: user \t item \t rating \t timestamp ---
ratings = pd.read_csv(os.path.join(R,"u.data"), sep="\t",
                      names=["user","item","rating","ts"])   # 读 10 万条评分
# --- 电影元信息 u.item: id|title|date|...|19 个类型 0/1 列 ---
GENRES = ["unknown","Action","Adventure","Animation","Children","Comedy","Crime",
          "Documentary","Drama","Fantasy","FilmNoir","Horror","Musical","Mystery",
          "Romance","SciFi","Thriller","War","Western"]        # 19 个类型名 / genre names
cols = ["item","title","date","video","url"] + GENRES          # u.item 的列名
movies = pd.read_csv(os.path.join(R,"u.item"), sep="|", encoding="latin-1",
                     header=None, names=cols)                  # 读电影表
movies = movies.set_index("item")                              # 用 item id 作索引，便于查表

print("评分 ratings:", ratings.shape, "| 电影 movies:", movies.shape)
print("示例电影 / sample:", movies.loc[1,"title"],
      "->", [g for g in GENRES if movies.loc[1,g]==1])          # 打印 1 号电影及其类型


**中文**：每部电影用一个 19 维的 0/1 向量表示它属于哪些类型（multi-hot）。比如《Toy Story》是 Animation + Children + Comedy。最朴素的相似度就是直接对这些 0/1 向量算余弦。但这样有个问题——**热门类型权重过大**：Drama 出现在几乎一半电影里，区分度低；而 FilmNoir 很罕见，一旦匹配上应该是强信号。
**English**: Each movie is a 19-dim 0/1 vector of which genres it belongs to (multi-hot). E.g. *Toy Story* = Animation + Children + Comedy. The naivest similarity is cosine over these 0/1 vectors. But there is a catch — **popular genres dominate**: *Drama* appears in nearly half the movies (low discriminative power), while *FilmNoir* is rare, so a match on it should be a strong signal.

**中文**：这正是 **TF-IDF** 要解决的问题：用 **IDF（逆文档频率）** 给罕见类型加权、给烂大街的类型降权。把"类型"当作"词"、把"电影"当作"文档"，content-based 就变成了一个文本相似度问题。
**English**: This is exactly what **TF-IDF** fixes: **IDF (inverse document frequency)** up-weights rare genres and down-weights ubiquitous ones. Treat each "genre" as a "word" and each "movie" as a "document" — content-based becomes a text-similarity problem.

$$\text{idf}(g) = \log\frac{N}{1+\text{df}(g)},\qquad \text{tfidf}(m,g)=\text{tf}(m,g)\cdot \text{idf}(g)$$

**中文**：其中 $N$ 是电影总数，$\text{df}(g)$ 是含类型 $g$ 的电影数，$\text{tf}(m,g)$ 是电影 $m$ 是否有类型 $g$（这里就是 0/1）。分母 $1+\text{df}$ 防止除零；某类型越普遍 $\text{df}$ 越大、$\text{idf}$ 越小。
**English**: $N$ = total movies, $\text{df}(g)$ = number of movies having genre $g$, $\text{tf}(m,g)$ = whether movie $m$ has genre $g$ (0/1 here). The $1+\text{df}$ denominator avoids division by zero; the more common a genre, the larger $\text{df}$ and the smaller $\text{idf}$.


In [ ]:

# ============================================================
# 第 1 步：用 TF-IDF 把每部电影编码成"类型向量" / Encode movies as TF-IDF genre vectors
# 中文：我们手写 TF-IDF（而非直接调库），这样能看清每一步在做什么。
# English: We hand-code TF-IDF (instead of calling a library) to see every step.
# ============================================================
G = movies[GENRES].values.astype(float)            # (n_movies, 19) 的 0/1 类型矩阵 / genre matrix
N = G.shape[0]                                      # 电影总数 N / number of movies

df = G.sum(axis=0)                                  # 每个类型的文档频率 df(g)：在多少部电影出现
idf = np.log(N / (1.0 + df))                        # IDF：罕见类型权重高 / rare genre -> high weight
tfidf = G * idf                                     # 广播相乘 (n,19)*(19,) -> (n,19)，得到 TF-IDF 矩阵

# 看看哪些类型权重最高/最低（越罕见 idf 越大）/ which genres weigh most/least
order = np.argsort(idf)                             # 按 idf 升序排序的索引
print("最常见(低权重) / most common (low weight):",
      [(GENRES[i], round(idf[i],2)) for i in order[:3]])     # idf 最小的 3 个
print("最罕见(高权重) / rarest (high weight):    ",
      [(GENRES[i], round(idf[i],2)) for i in order[-3:]])    # idf 最大的 3 个


**中文**：有了每部电影的 TF-IDF 向量，"两部电影有多像"就用**余弦相似度**衡量——即两个向量夹角的余弦。它只看方向不看长度，所以不受"一部电影标了几个类型"的影响。
**English**: With a TF-IDF vector per movie, "how similar are two movies" is measured by **cosine similarity** — the cosine of the angle between the two vectors. It looks at direction not magnitude, so it is unaffected by "how many genres a movie is tagged with."

$$\cos(\mathbf{a},\mathbf{b}) = \frac{\mathbf{a}\cdot\mathbf{b}}{\|\mathbf{a}\|\,\|\mathbf{b}\|}$$

**中文**：分子是点积（共同类型的加权重叠），分母是两个向量的模长之积（归一化）。取值 $[0,1]$（这里向量非负），越接近 1 越像。
**English**: The numerator is the dot product (weighted overlap of shared genres), the denominator is the product of the two norms (normalization). Range $[0,1]$ here (non-negative vectors); closer to 1 means more similar.


In [ ]:

# ============================================================
# 第 2 步：余弦相似度矩阵 / Cosine similarity matrix
# 中文：先把每行 L2 归一化，则任意两行点积 = 余弦。整个相似度矩阵一次矩阵乘搞定。
# English: L2-normalize each row; then dot product of any two rows = cosine.
#          The whole pairwise matrix is a single matmul.
# ============================================================
norm = np.linalg.norm(tfidf, axis=1, keepdims=True)   # 每部电影向量的模长 (n,1)
norm[norm==0] = 1e-9                                   # 防止除零（极少数无类型电影）/ avoid div-by-0
unit = tfidf / norm                                    # 归一化为单位向量 / unit vectors (n,19)
S = unit @ unit.T                                      # (n,n) 余弦相似度矩阵 / cosine sim matrix

# 用电影名查 item id 的小工具 / helper: title -> item id
title2id = {t:i for i,t in movies["title"].items()}   # 标题到 id 的映射

def similar_movies(title, k=5):
    """中文：返回与给定电影最相似的 k 部 / English: top-k most similar movies."""
    mid = title2id[title]                              # 取该电影的 item id
    pos = movies.index.get_loc(mid)                    # id -> 矩阵中的行号 / row position
    sims = S[pos].copy()                               # 该电影与所有电影的相似度
    sims[pos] = -1                                     # 排除自己 / exclude itself
    top = np.argsort(sims)[::-1][:k]                   # 取相似度最高的 k 个行号
    return [(movies.iloc[j]["title"], round(sims[j],3)) for j in top]

print("与《Toy Story (1995)》最像的电影 / most similar to Toy Story:")
for t,s in similar_movies("Toy Story (1995)"): print(f"  {s:.3f}  {t}")


**中文**：上面是"物品到物品"的相似推荐（看了 A 推 B）。但真正的推荐要落到**用户**身上：给定一个用户的历史评分，给他推他可能喜欢的新电影。做法是构造**用户画像（user profile）**——把他打过高分的电影的 TF-IDF 向量**按评分加权求和**，得到一个"这个用户的口味向量"，再拿它和所有电影算相似度。
**English**: The above is item-to-item ("watched A → recommend B"). Real recommendation targets a **user**: given their rating history, suggest new movies. We build a **user profile** — a rating-weighted sum of the TF-IDF vectors of movies they liked — a single "taste vector," then score all movies by similarity to it.

$$\mathbf{p}_u = \sum_{i \in \text{liked}(u)} (r_{ui}-\bar r)\;\mathbf{v}_i$$

**中文**：$\mathbf{v}_i$ 是电影 $i$ 的 TF-IDF 向量，$r_{ui}$ 是用户 $u$ 对它的评分，减去全局均值 $\bar r$ 让"低于平均的评分"产生**负贡献**（push away），而非都当成喜欢。
**English**: $\mathbf{v}_i$ is movie $i$'s TF-IDF vector, $r_{ui}$ is user $u$'s rating, and subtracting the global mean $\bar r$ makes below-average ratings contribute **negatively** (push away) instead of treating every rating as a "like."


In [ ]:

# ============================================================
# 第 3 步：用户画像 + 个性化推荐 / User profile + personalized recs
# ============================================================
mean_r = ratings["rating"].mean()                      # 全局平均分 ~3.5 / global mean rating
id2pos = {mid:p for p,mid in enumerate(movies.index)}  # item id -> 行号 / id to row position

def recommend_for_user(uid, k=5):
    hist = ratings[ratings.user==uid]                  # 该用户的全部评分记录 / user's ratings
    seen = set(hist.item)                              # 已看过的电影集合（推荐时要排除）
    # 构造口味向量：每部看过的电影 TF-IDF 向量 × (评分-均值)，再求和
    profile = np.zeros(unit.shape[1])                  # 初始化 19 维口味向量 / taste vector
    for _,row in hist.iterrows():
        p = id2pos[row["item"]]                           # 电影行号
        profile += (row["rating"] - mean_r) * unit[p]     # 评分居中后加权累加 / weighted accumulate
    n = np.linalg.norm(profile)                        # 归一化口味向量 / normalize
    profile = profile/n if n>0 else profile
    scores = unit @ profile                            # 用口味向量给所有电影打分 (n,) / score all
    rec = []
    for j in np.argsort(scores)[::-1]:                 # 从高分到低分扫描 / high to low
        mid = movies.index[j]
        if mid not in seen:                            # 跳过已看过的 / skip seen
            rec.append((movies.loc[mid,"title"], round(scores[j],3)))
        if len(rec)==k: break
    return hist, rec

hist, rec = recommend_for_user(1, k=5)                 # 给 1 号用户推荐 / recommend for user 1
top_liked = hist.sort_values("rating",ascending=False).head(3)
print("用户1 打高分的电影 / user 1's top-rated:")
for _,r in top_liked.iterrows(): print(f"  {r.rating}星  {movies.loc[r['item'],'title']}")
print("\n基于内容为用户1推荐 / content-based recs for user 1:")
for t,s in rec: print(f"  {s:.3f}  {t}")


**中文**：现在做一个**严肃的离线评估**——光看"推荐列表顺眼"不算数，找工作面试时面试官一定会追问"你怎么证明它有效？"。我们用**留一法（leave-one-out）**的思路：按时间把每个用户的评分切成训练/测试，只用训练评分建用户画像，看测试集里他真正打高分的电影能不能被排到前面。
**English**: Now a **serious offline evaluation** — "the list looks reasonable" is not evidence; an interviewer will ask "how do you prove it works?" We use a **temporal train/test split** per user: build the profile from training ratings only, then check whether movies the user actually rated highly in the held-out test set get ranked near the top.

**中文**：指标用 **Precision@K** 和 **Recall@K**：在推荐的前 K 个里，有多少是用户测试集中真正喜欢（≥4 星）的（precision），以及测试集中所有喜欢的电影里有多少被我们命中（recall）。这两个指标我们会在 15.11 系统展开，这里先建立直觉。
**English**: Metrics: **Precision@K** and **Recall@K** — among the top-K recommendations, how many are truly liked (≥4 stars) in the user's test set (precision), and of all liked test movies how many we hit (recall). We expand these systematically in 15.11; here we build intuition.


In [ ]:

# ============================================================
# 第 4 步：离线评估 Precision@K / Recall@K，并与"推荐最热门"基线对比
# Offline eval vs. a popularity baseline
# 中文：基于内容的模型必须打败"无脑推最热门电影"这个强基线，否则没有存在意义。
# English: content-based must beat the strong "just recommend the most popular" baseline.
# ============================================================
def eval_split(K=10):
    # 按时间戳给每个用户切 80/20：早期当训练、近期当测试 / temporal 80/20 split per user
    ratings_sorted = ratings.sort_values("ts")
    train_idx, test_idx = [], []
    for uid,grp in ratings_sorted.groupby("user"):
        cut = int(len(grp)*0.8)                        # 前 80% 训练
        train_idx += list(grp.index[:cut]); test_idx += list(grp.index[cut:])
    tr = ratings.loc[train_idx]; te = ratings.loc[test_idx]

    # 热门基线：训练集中被评分次数最多的电影（按人气降序）/ popularity baseline
    pop = tr.groupby("item").size().sort_values(ascending=False)
    pop_list = list(pop.index)

    # 预计算每个用户的训练画像 / precompute training profiles
    def profile_for(grp):
        v = np.zeros(unit.shape[1])
        for _,r in grp.iterrows(): v += (r["rating"]-mean_r)*unit[id2pos[r["item"]]]
        nn=np.linalg.norm(v); return v/nn if nn>0 else v

    cb_p=cb_r=pp=pr=0.0; nuser=0
    for uid,tgrp in te.groupby("user"):
        liked = set(tgrp[tgrp.rating>=4].item)         # 测试集中真正喜欢的(≥4星) / relevant items
        if not liked: continue                         # 没有正样本就跳过 / skip if none
        tr_u = tr[tr.user==uid]; seen=set(tr_u.item)   # 训练中看过的要排除
        if len(tr_u)==0: continue
        # --- content-based 推荐前K ---
        prof = profile_for(tr_u); sc = unit@prof
        rec=[]
        for j in np.argsort(sc)[::-1]:
            mid=movies.index[j]
            if mid not in seen: rec.append(mid)
            if len(rec)==K: break
        hit=len(set(rec)&liked)
        cb_p += hit/K; cb_r += hit/len(liked)
        # --- popularity 推荐前K ---
        prec=[m for m in pop_list if m not in seen][:K]
        h2=len(set(prec)&liked)
        pp += h2/K; pr += h2/len(liked)
        nuser+=1
    return (cb_p/nuser, cb_r/nuser, pp/nuser, pr/nuser, nuser)

cbP,cbR,pP,pR,nu = eval_split(K=10)
print(f"评估用户数 / users evaluated: {nu}")
print(f"{'方法/method':<22}{'Precision@10':>14}{'Recall@10':>12}")
print(f"{'Content-based':<22}{cbP:>14.4f}{cbR:>12.4f}")
print(f"{'Popularity baseline':<22}{pP:>14.4f}{pR:>12.4f}")


**中文**：结果很可能让你意外——**纯基于内容（只用 19 个类型标签）往往打不过"推荐最热门"基线**，甚至更差。这不是 bug，而是一个极其重要的、面试中能讲出来就加分的真相：
**English**: The result will likely surprise you — **pure content-based (only 19 genre tags) often fails to beat the popularity baseline**, sometimes losing outright. This is not a bug; it is a crucial truth that scores points in interviews:

**中文**：
1. **特征太弱**：19 个类型标签信息量很低，无法刻画"剧本质量、演员、节奏"这些真正决定喜好的因素。content-based 的天花板由特征质量决定。
2. **热门基线非常强**：推荐里有个著名现象——大多数人都会看热门电影，所以"推最热门"在 precision 上极难超越。这是为什么工业界推荐系统几乎都把 popularity 当作必须超越的 sanity baseline。
3. **过度专门化**：content-based 只会推"同类型"，多样性差，容易错过用户其实也喜欢的跨类型作品。

**English**:
1. **Features are weak**: 19 genre tags carry little information — they cannot capture script quality, cast, or pacing, which truly drive taste. Content-based is capped by feature quality.
2. **Popularity is a strong baseline**: most people watch popular movies, so "recommend the most popular" is famously hard to beat on precision — which is why industry always uses popularity as a sanity baseline to beat.
3. **Over-specialization**: content-based only recommends the same genre, hurting diversity and missing cross-genre items the user would also enjoy.

> 💼 **实战视角 / Practical angle**
> **中文**：所以工业界很少**单独**用基于内容，而是把它当作**冷启动兜底**（新物品没人评分时唯一能用的信号）和**召回的一路**（多路召回里的"内容召回"），再交给后面的协同过滤/排序模型。下一节 15.2 我们就引入"别人的行为"——协同过滤，看它能否打败热门基线。
> **English**: Industry rarely uses content-based **alone**; it serves as a **cold-start fallback** (the only signal when a new item has no ratings) and as **one retrieval channel** in multi-channel recall, with collaborative filtering / ranking models downstream. In 15.2 we bring in "other people's behavior" — collaborative filtering — and see whether it beats popularity.

---
### 小结 / Summary
- **中文**：基于内容 = 物品特征相似度（TF-IDF + 余弦）；用户画像 = 评分加权的特征向量和。
- **English**: Content-based = item-feature similarity (TF-IDF + cosine); user profile = rating-weighted sum of feature vectors.
- **中文**：优点无物品冷启动 + 可解释；缺点用户冷启动 + 过度专门化 + 受限于特征质量。
- **English**: Pros: no item cold-start + explainable; cons: user cold-start + over-specialization + feature-bound.
- **中文**：永远要和 **popularity 基线** 对比——这是推荐系统离线评估的"及格线"。
- **English**: Always compare against the **popularity baseline** — the "pass line" of offline recsys evaluation.
